# R parity and noted differences

`mboostrs` is validated against R `mboost` and `gamboostLSS` to tight
numerical tolerances. This chapter describes the parity tests and records
behavioral or numerical differences from the R implementations, with the
supporting examples and their effects on fitted results.

## Parity test procedure

The test suite runs the same model in R (via `rpy2`) and in `mboostrs` on shared
data and asserts agreement on the quantities that define a fit:

- the full **risk path** (length $m_{\text{stop}}+1$),
- the **fitted values** / predictions (link and response),
- the **selection sequence** and **coefficients**,
- offsets and nuisance parameters.

Typical tolerances are $10^{-6}$ relative, often $10^{-12}$–$10^{-14}$ for
offsets, gradients, and risk. R-parity tests are marked and skipped where R is
unavailable; the core also carries R-independent unit tests (golden values and
finite-difference gradient checks) so the numerics are protected without R.

## Validation approach

Each numerical difference is checked against the family formulas and, where
possible, against R using the same intermediate values. Finite-difference
gradient checks verify analytical derivatives; parity tests check the shared
boosting machinery separately from family-specific calculations.

## Noted differences

Each difference isolates a specific behavior in R, with a snippet that
reproduces it from a clean session.

The examples were verified with `mboost` 2.9-13,
`gamboostLSS` 2.2-0, `gamlss.dist` 6.1-1, and R 4.5.1. They distinguish
observed implementation differences from behavior whose cause remains unknown.

### LSS offset and gradient differences

| Family | Parameter | Observed R behavior | Practical impact | mboostrs behavior |
|---|---|---|---|---|
| `LogLogLSS` | σ | Offset calls `pnorm` (LogNormalLSS legacy) instead of `plogis` | Censored obs get a slightly wrong offset $\rightarrow$ 0.5–1% risk difference on typical survival data | Sequential offset minimizes the correct log-logistic negative log-likelihood (NLL) |
| `WeibullLSS` | μ | Identity link, but offset returns `log(ȳ)` via an extraneous `offset="scalar"` recomputation | Initial μ is ∼1.5× too low; boosting needs ∼5 extra iterations to converge | Offset solver returns μ on the original scale |
| `ZINBLSS` | ν (zero-inflation) | `dNBI(0, …)` recycling causes the first row's zero-inflation density to be reused for every observation | Order-dependent: shuffling the data changes the gradient by up to 10% on sparse data | Per-observation `dNBI` evaluation via `gamboostLSS::dNBI` |

## LogLogLSS sigma offset for censored observations

```r
library(gamboostLSS)
set.seed(1)
n <- 50; y <- Surv(rexp(n), rbinom(n, 1, 0.7))
fit <- glmboostLSS(y ~ 1, families = LogLogLSS(), control = boost_control(mstop = 1))
# σ-offset for a censored obs uses pnorm(log(y)) instead of plogis(log(y)):
loss <- families.gamboostLSS(LogLogLSS())$loss
y_cens <- 2.0; delta <- 0  # censored
loss(sigma = 0.5, y = y_cens, delta = delta)  # calls pnorm, not plogis
```

```{list-table}
:header-rows: 1

* - Where
  - Observed behavior in R
  - `mboostrs`
* - `gamboostLSS::LogLogLSS` σ offset
  - The σ `loss` pairs a logistic density (`dlogis`) with the standard-normal
    survival (`pnorm`) for censored observations, where the log-logistic survival
    (`plogis`) is required. Confined to the risk and hence the offset; both
    `ngradient`s compute the correct log-logistic gradient in closed form
  - minimizes the correct log-logistic negative log-likelihood (NLL)
```

Three checks pin this down. `integrate(dlogis, eta, Inf)` equals
`plogis(eta, lower.tail = FALSE)` at every `eta` and differs from
`pnorm(eta, lower.tail = FALSE)` by a factor of five at `eta = 2`. The same
package is internally inconsistent: `LogNormalLSS` uses `dnorm`/`pnorm` and
`LogLogLSS`'s **μ** uses `dlogis`/`plogis`, both consistent, while `LogLogLSS`'s
**σ** mixes them — its `logSw` line is character-identical to
`LogNormalLSS$sigma`'s. And on 4000 observations from a true log-logistic with
40% censoring, the consistent loss recovers μ = 1.5004 against a truth of 1.5 and
matches `survival::survreg(dist = "loglogistic")` to four decimals, while the
loss as coded returns μ = 1.7717.

## WeibullLSS mu offset

The observed `WeibullLSS` offset changes with the time scale of the response.
The source path that produces this value remains unresolved. The following
example reproduces the values from a clean R session:

```r
library(gamboostLSS); library(survival)
for (s in c(5, 0.05)) {
  set.seed(7); n <- 3000
  t <- rweibull(n, shape = 2, scale = s); y <- Surv(t, rep(1, n))
  fit <- glmboostLSS(y ~ 1, families = WeibullLSS(), control = boost_control(mstop = 1))
  cat(sprintf("scale=%-5g max(t)=%9.5f interval=[0,%9.5f] mu offset=%+12.6f survreg mu=%+8.4f\n",
      s, max(t), max(t), fit$mu$offset,
      coef(survreg(Surv(t, rep(1, n)) ~ 1, dist = "weibull"))[1]))
}
# scale=5     max(t)= 13.19806 interval=[0, 13.19806] mu offset=   +0.401470 survreg mu= +1.6145
# scale=0.05  max(t)=  0.13198 interval=[0,  0.13198] mu offset=   -9.788913 survreg mu= -2.9906
```

μ has an identity link and enters the loss as `eta = (log t - mu)/sigma`, so it
appears to be on the log-time scale, while the offset's search interval
`c(0, max(y[, 1]))` is on the time scale. In the second case the returned value
lies outside that interval, which `optimize` cannot produce, so a different
branch must run — but where its `mu` is set could not be determined. Both runs
reproduce from clean sessions.

An offset is a starting value that subsequent boosting updates can change, but
the effect of this R behavior on convergence has not been measured. `mboostrs`
computes its Weibull mu offset by minimizing the Weibull negative
log-likelihood directly.

## `gamlss.dist::dNBI` vectorization in `ZINBLSS`

```r
library(gamlss.dist)
dNBI(0, mu = c(0.5, 1.0, 2.0), sigma = 0.5)
# Returns length 1, not 3 — the trailing ifelse(x < 0, ...) collapses the vector
# to a scalar. ZINBLSS workers see the same zero-inflation probability for all obs.
```

```{list-table}
:header-rows: 1

* - Where
  - Observed behavior in R
  - `mboostrs`
* - `gamlss.dist` `dNBI` in `ZINBLSS`
  - `dNBI(0, mu_vector, sigma)` collapses to length 1 via a trailing
    `ifelse(x<0,…)`, so the ZINB gradient recycles the **first** observation's
    zero-density across all rows (order-dependent)
  - computes `dNBI(0, mu_i, sigma)` per observation
```

## Multinomial softmax denominator in `mboost`

`mboost` bounds the predictor before exponentiating instead of subtracting a
per-row maximum. Its `ngradient` uses:

```r
f <- pmin(abs(f), 36) * sign(f)
p <- matrix(exp(f), ncol = length(lev) - 1)
p <- as.vector(p / (1 + rowSums(p)))
```

With no subtracted maximum, `1 + rowSums(exp(f))` is the correct
baseline-category denominator because the reference class has logit 0 and
`exp(0) = 1`. In the max-subtracting formulation used by `mboostrs`, the
reference term is `exp(-m)`. The formulations produce the same probabilities;
they differ in how they stabilize exponentiation. See
[](kronecker_multinomial.ipynb) for why `mboostrs` subtracts a maximum.




(response-units)=
## Response units

Gradient boosting starts from a constant and adds, at each step, `nu` times a
base learner's fit to the negative gradient of the loss. For squared error the
negative gradient is the residual, so measuring the response in other units
rescales every step by the same factor and the fit is the same model in the new
units. For several other losses that is not so, and R `mboost` and
`gamboostLSS` fit the response in the units it is given: the same data in
dollars and in thousands of dollars give different models at the same
`mstop`. mboostrs divides the response by a scale before these fits and maps
the model back, so the model is the same in any units. The table lists where
mboostrs and R differ for that reason.

| Fit | Negative gradient | Step size in response units (R) | mboostrs default |
|---|---|---|---|
| Distributional (`gamboostLSS`, `glmboostLSS`), stabilized | divided by its own spread | about `nu` | `response_scale="auto"` |
| Distributional, unstabilized `GaussianLSS` | `(y - mu) / sigma^2` for `mu` | `nu` times the residual over `sigma^2`: shrinks as the units grow | `response_scale="auto"` |
| `Quantile` (R `QuantReg`) | `tau` or `tau - 1` | about `nu * tau` | `response_scale="auto"` |
| `Laplace` | `sign(y - f)` | about `nu` | `response_scale="auto"` |
| `Gaussian`, `Expectile`, `Huber` | proportional to the residual (Huber's threshold is stated in response units) | scales with the response | unchanged |
| Counts, binary, proportions, compositions | no units to change | | unchanged |

### Distributional fits

With `response_scale="auto"` a distributional fit divides the response by
`s`, the weighted standard deviation of the response it fits, boosts `y / s`,
and maps each distribution parameter's linear predictor back exactly:
`eta = s * eta_s` for an identity-link location (the Gaussian, Student t,
logistic, Gumbel and Johnson SU means, the BCT and BCPE medians), and
`eta = eta_s + c log(s)` for a log-link parameter proportional to `s^c` (a
Gaussian or Student t scale, `c = 1`; the gamma, inverse Gaussian, Pareto and
BCCG means, `c = 1`; the inverse Gaussian sigma, `c = -1/2`; the known-variance
Gaussian's excess variance, `c = 2`) and for the log-time location of the
survival families. Shape parameters (degrees of freedom, skewness, kurtosis,
the gamma shape) do not change. Every stored coefficient is multiplied by the
same factor, and the risk is shifted by `log(s)` per row with a density on the
response, so predictions, coefficients, `logLik`, `gaic` and saved models are
all in the original units. The map is checked, for every family it covers,
against the family's own negative log-likelihood at three scales
(`every_scale_map_reproduces_the_likelihood_in_original_units` in the Rust
core). Count, binary, beta and Dirichlet families have no units and are not
scaled.

Because the scale is computed from the rows each fit sees, a cross-validation
fold, a weighted refit and a fit on the same rows as a subset take the same
scale and describe the same path. `response_scale=None` (or `"none"`) fits the
raw response, which is R gamboostLSS's path, and a positive number fixes the
scale.

The difference from R is the boosting path, not the model class. On California
housing (median house value in dollars, SD about 115,000) a stabilized
GaussianLSS fitted by R's rule barely moves its mean from the offset: the test
RMSE was 114,622, the response SD, after 200 and after 1,000 steps, and the
tuned model's CRPS (62,111) was worse than that of the unconditional
distribution of the training responses (63,083). The scaled fit reached CRPS
33,047 on the same split. A model fitted by R on a response whose SD is near 1
is close to the scaled fit; one whose SD is far from 1 is not.

Scaling does not remove the instability of an unstabilized GaussianLSS fit:
its `mu` step is proportional to `1 / sigma^2`, so rows whose spread is far
below the response's overall spread take large steps. On the unit scale this
can overshoot where the raw scale (with an SD above 1) did not; the divergence
error names `stabilization="MAD"` and a family for the response's shape as the
remedies.

### Quantile and Laplace fits

The check loss and the absolute loss have a negative gradient whose size does
not depend on the response, so each step moves the fit by about `nu * tau` (or
`nu`) in response units. Fitting `y / s` and mapping back is the same as
multiplying the gradient by `s`: the sign of a residual does not change when
the response is divided by `s`, a base learner's fit is linear in its target,
and the offset (a weighted quantile) already scales with the units. mboostrs
does exactly that with `response_scale="auto"` (the default for `Quantile` and
`Laplace`). On California housing in dollars, the tuned median regression's
pinball loss was 43,787 by R's rule and 21,537 on the response divided by its
SD, and the 0.9-quantile's was 53,700 against 12,772 (both with R's other
defaults, `qoffset = 0.5` and `nu = 0.1`). With the current defaults the fit
in dollars and the fit on the scaled response give identical losses, 21,410
and 12,922. `Quantile(..., response_scale=None)` and
`Laplace(response_scale=None)` are R's raw-units fits.

The Laplace offset is R's Brent search for the weighted median, whose stopping
tolerance is absolute, so in R it is found more finely, relative to the data,
in larger units. Under scaling the search runs on `y / s` and its result is
multiplied by `s`. With an even number of rows every value between the two
middle observations minimises the absolute loss, and the search may stop
anywhere in that stretch; the scaled fits in different units agree up to that
choice (0.15% of the response SD on the test's 300 rows).

### Quantile defaults

Three defaults of `Quantile` differ from R `mboost`'s `QuantReg`. The offset
starts at the `tau`-quantile of the response, not at the median (`qoffset`
defaults to `tau`; R's is 0.5 whatever `tau`). A fit given no `control`, and
`tune_mstop` given no `nu`, take steps of `nu = 0.3` rather than 0.1. And the
response is scaled as above. The step length was chosen on 18 quantile fits
(six OpenML-CTR23 data sets at `tau` 0.1, 0.5 and 0.9, each tuned by
`tune_mstop` with five folds on the response divided by its SD):

| Setting | Test pinball loss against R's (geometric mean) | Worst task | Median steps | Fits stopped at the 10,000-step limit | Median seconds |
|---|---|---|---|---|---|
| `qoffset=0.5`, `nu=0.1` (R) | 1.000 | | 3,691 | 8 of 18 | 7.2 |
| `qoffset=0.5`, `nu=0.3` | 0.984 | 1.023 | 1,668 | 6 of 18 | 2.7 |
| `qoffset=tau`, `nu=0.1` | 0.988 | 1.002 | 5,357 | 8 of 18 | 6.5 |
| `qoffset=tau`, `nu=0.3` | 0.986 | 1.023 | 1,510 | 6 of 18 | 2.6 |

The 18 fits are one split each, so differences of one or two per cent are
within their noise; the gain is in time. `Quantile(tau, qoffset=0.5,
response_scale=None)` with `boost_control(mstop, nu=0.1)` reproduces
`QuantReg`.

### GammaLSS offset search

R's `GammaLSS` finds its `mu` offset by Brent's method on the log mean over
`[0, max(y^2)]` and its `sigma` (log shape) offset over `[0, max(y)]`. A response
whose mean is below 1 needs a negative log mean, and one more skewed than an
exponential needs a log shape below 0; R returns the boundary of the interval
for both. mboostrs opens the lower end of the interval in exactly those cases
(when the risk still decreases at the boundary) and keeps R's interval
otherwise, so every offset R's interval contains is unchanged. The case is
common under response scaling: a positive response divided by its SD has mean
below 1 whenever its coefficient of variation exceeds 1.

### Unseen group levels in models from `mb.fit`

A `brandom` or factor term has no coefficient for a level its training rows did
not contain. `predict` refuses such a row by default, in mboostrs as in R. A
model returned by `mb.fit` predicts it as a member of an average group
instead, and records `unseen_levels: "average"` in its trail. `mb.fit` reads a
whole-number column whose values behave as labels as a grouping, and on new
rows such a column routinely carries new codes: on OpenML video_transcoding
every test split carried codes the training rows did not, and every
prediction was refused. `predict(..., unseen_levels="error")` still refuses,
and a model fitted directly keeps the refusal as its default.

### Distributional adoption in `autospec`

These are mboostrs's own procedures, with no R counterpart, recorded here
because they change which model `mb.fit` returns.

- `autospec` adopts a distributional family when its held-out log score beats
  the mean family's by 2% of the mean family's score and by two paired fold
  standard errors. A log score shifts by `log(c)` when the response is measured
  in units `c` times smaller, so the 2% bar depended on the units (about 0.25
  nats per row for California housing in dollars, 0.02 for the same response in
  units of its SD). The bar is now read on the score of the response divided by
  its SD.
- The spread screen regresses `log(residual^2)` on each column linearly, so a
  spread that depends on a product or a threshold of two predictors leaves
  every slope near zero. When the screen finds no column, `autospec` now offers
  GaussianLSS with a joint depth-3 tree over the numeric predictors as the scale
  equation, decided by the same held-out gates.
- `autospec(lss_trees=...)` offers an adopted distributional model a joint tree
  (depths 3 and 6) in every parameter, adopted under the same gates.


In [ ]:
import numpy as np
import mboostrs as mb

rng = np.random.default_rng(3)
x = rng.uniform(-1, 1, 300)
y = 1.0 + np.sin(2 * x) + np.exp(-1.0 + 0.6 * x) * rng.standard_normal(300)
spec = {"mu": "y ~ bbs(x)", "sigma": "y ~ bbs(x)"}


def mean_rmse(model, response):
    mu = np.asarray(model.predict(type="response")["mu"], dtype=float)
    return float(np.sqrt(np.mean((response - mu) ** 2)))


rows = []
for unit in (1.0, 1e3, 1e5):
    data = {"x": x, "y": unit * y}
    for scale in ("auto", None):
        fit = mb.gamboostLSS(spec, data, family=mb.GaussianLSS(stabilization="MAD"),
                             control=mb.boost_control(150, 0.1), response_scale=scale)
        rows.append({"units": unit, "response_scale": str(scale),
                     "RMSE of the mean / unit": round(mean_rmse(fit, data["y"]) / unit, 4)})
display(rows)

With the scaled fit the mean's error, in the response's own units, is the
same whatever the units (0.408 of a unit). With R's raw-units path it is the
same only on the unit scale; in units of 1,000 and of 100,000 the mean has
barely left its offset after 150 steps, and its error (0.82 to 0.83 of a unit)
is close to the response's own standard deviation.

### Verifying the `mboostrs` softmax denominator

This needs no R: it is a claim about arithmetic within one formulation. If a
softmax is written by subtracting a per-observation maximum $m_i$, then the
reference class — whose logit is a literal 0 — contributes $e^{0-m_i} = e^{-m_i}$
to the denominator, not 1. Substituting 1 there is only correct when no maximum
has been subtracted. The check evaluates both expressions in a case where they disagree, and reads back which one the engine implements via
`family_diagnostics`.

In [1]:
import numpy as np
from mboostrs._rust import family_diagnostics

f = np.array([1.5, -0.5])  # one observation, K-1=2 class logits
m = max(0.0, f.max())      # the shift mboostrs subtracts for stability

# With a subtracted maximum, the reference class's literal logit of 0 contributes
# exp(0 - m) = exp(-m).
shifted_denom = np.exp(-m) + np.sum(np.exp(f - m))
# Keeping a literal 1 alongside shifted terms mixes the two conventions.
mixed_denom = 1.0 + np.sum(np.exp(f - m))
print(f"m = {m}")
print(f"consistently shifted denominator: {shifted_denom:.4f}")
print(f"mixed (1 + shifted terms):        {mixed_denom:.4f}")
assert abs(shifted_denom - mixed_denom) > 0.1  # they disagree whenever m > 0

# Which does mboostrs use? Read P(y=k) back out of the engine's own gradient
# (u = y - P, so P = y - u) and compare against both candidates.
y = np.array([1.0, 0.0])
w = np.array([1.0])  # one weight per observation
_, grad, _ = family_diagnostics("multinomial", {"n_classes": 3.0}, y, f, w)
p_a_engine = y[0] - grad[0]
p_a_shifted = np.exp(f[0] - m) / shifted_denom
p_a_mixed = np.exp(f[0] - m) / mixed_denom
print(f"engine's P(A):           {p_a_engine:.6f}")
print(f"consistently shifted:    {p_a_shifted:.6f}")
print(f"mixed conventions:       {p_a_mixed:.6f}")

assert abs(p_a_engine - p_a_shifted) < 1e-10
assert abs(p_a_engine - p_a_mixed) > 1e-3


m = 1.5
consistently shifted denominator: 1.3585
mixed (1 + shifted terms):        2.1353
engine's P(A):           0.736125
consistently shifted:    0.736125
mixed conventions:       0.468311


The two expressions differ by 57% here, and the engine matches the consistently
shifted one to floating-point precision. This says nothing about mboost, which
does not subtract a maximum: it clamps the predictor to $\pm 36$ and uses
$1 + \sum_k e^{f_k}$, which is correct for an unshifted denominator. The check
confirms that `mboostrs`'s own stabilization is internally consistent, using the
same `family_diagnostics` binding as [](../methodology/multinomial).

## JSULSS centered parameterization

The Rust implementation matches the centred parameterization of
`gamlss.dist::JSU()` (Johnson's SU distribution), which applies centreing constants ($c$, $\omega$, $w$)
so that the distribution parameters $\mu$ and $\sigma$ correspond to the
population mean and standard deviation. Without these constants (the uncentred
JSU), coefficients differ from R even though the density and gradients are
self-consistent. Coefficient-level parity against R's `gamboostLSS` is not
yet verified, though the mathematical density and gradients are independently
validated by finite-difference checks.

## Cross-validation stopping excludes iteration zero

R's `mstop(cvrisk_object)` never returns 0 (the offset-only model), even when
the risk curve is monotonically increasing. `mboostrs`'s `optimal_mstop()`
restricts the argmin search to start from index 1, matching R's behavior and
preventing selection of an empty model when signals are weak.

## `mboost` HingeLoss offset

`HingeLoss`'s `offset` returns the AdaBoost/exponential-loss log-odds, which does
not minimize the hinge risk. The offset function is the same object as
`AdaExp`'s — `identical(deparse(HingeLoss()@offset), deparse(AdaExp()@offset))`
is `TRUE`. The hinge risk over a constant $f = c$ is piecewise linear on
$[-1, 1]$ and strictly worse outside, so its minimizer is a boundary point
$\operatorname{sign}(W_+ - W_-)$, never an interior log-odds:

```r
library(mboost)
h <- getNamespace("mboost")$HingeLoss()
y <- c(rep(1, 5), rep(-1, 3)); w <- rep(1, 8)
f <- h@offset(y, w)                 # 0.2554128
h@risk(y, rep(f, 8), w)             # 3.744587
h@risk(y, rep(1, 8), w)             # 3.000000  <- the argmin
```

`HingeLoss` is not exported and no function in the mboost namespace references
it, so this difference is unreachable through a documented route. The offset
is a starting value that later boosting updates can change. For `Gaussian`,
`Huber`, `Poisson`, and `AdaExp`, the offset minimizes the risk. For `Laplace`,
the risk is flat between order statistics; the reported offset and a reference
minimizer differ by $7\times 10^{-15}$. `mboostrs` reproduces the R offset.

## Floating-point path divergence

Floating-point rounding can produce different paths even when two
implementations use equivalent formulas:

- **GAMBoost weight scaling drift**: When a constant weight vector is scaled by
  a factor (for example, 2), `mboostrs` may drift by ~1e-6 in fitted values and
  ~4e-5 in the risk path. The cause is floating-point path divergence in the
  `bbs` penalty Cholesky solve (`X' W X` entries scale linearly with the weight
  factor, changing the rounding of each triangular solve). GLMBoost (no penalty)
  is exact under weight scaling. The drift is a difference between two
  `mboostrs` fits of the same data, not a difference from R.

## Feature support and numerical limits

Current support and numerical limits include:

- **AIC for the multinomial / `%O%` model** is unavailable in both R and
  `mboostrs` (no hat matrix for the Kronecker learner).
- **`gamboostLSS` survival** in R can overflow on heavy-tailed data
  (`WeibullLSS`'s $e^{e}$ gradient); `mboostrs` has the same numeric limit and
  tests use a moderate response scale, matching R.
- Multinomial **newdata/truncated-mstop prediction** is fully supported in R
  for both linear `bols` and smooth `bbs` learners (bols and bbs are the only
  supported learner types for the Multinomial family).
- **R binding prediction for `center=TRUE` fits.** The R binding's prediction for models with `center=TRUE` (default for `bspatial`) does not correctly project new data into the spectrally-centred subspace: it rebuilds the raw B-spline design and checks column counts against the reduced coefficient vector, causing the prediction to use stale values. The Python binding avoids this by using `_predict_gamboost`, a Rust function that receives the `bbs_center_list` flags and reconstructs the centred projection internally — building each learner's basis once and aggregating coefficients across selected steps. This is independent of `LearnerStore`, which is used only for in-sample truncated-`mstop` prediction.

## See also

- [](../appendix/r_parity_matrix) — full feature matrix comparing R and mboostrs support.
- [](testing.ipynb) — how parity tests and finite-difference checks validate the implementation.